In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException, TimeoutException
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import pandas as pd
import re

BASE = "https://www.larazon.es"
SECCION = "https://www.larazon.es/opinion/"
OBJETIVO = 2500

# CONFIG SELENIUM

options = Options()
options.headless = False
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("--disable-gpu")
options.add_argument("--log-level=3")

def crear_driver():
    nuevo_driver = webdriver.Chrome(options=options)
    nuevo_driver.set_page_load_timeout(40)
    return nuevo_driver


driver = crear_driver()
cookies_aceptadas = False

def aceptar_cookies():
    global cookies_aceptadas

    if cookies_aceptadas:
        return

    try:
        botones = driver.find_elements(By.TAG_NAME, "button")
        for b in botones:
            texto = b.text.lower()
            if "acept" in texto or "agree" in texto:
                print("aceptando cookies...")
                b.click()
                time.sleep(2)
                cookies_aceptadas = True
                return
    except:
        pass


def get_soup(url, wait=4, intentos=3):
    global driver, cookies_aceptadas

    for intento in range(1, intentos + 1):
        try:
            print(f"\nabriendo: {url} | intento {intento}")

            driver.get(url)
            time.sleep(wait)
            aceptar_cookies()

            return BeautifulSoup(driver.page_source, "html.parser")

        except (ConnectionResetError, WebDriverException, TimeoutException) as e:
            print(f"fallo cargando pagina: {e}")

            try:
                driver.quit()
            except:
                pass

            print("reiniciando navegador...")
            driver = crear_driver()
            cookies_aceptadas = False

            time.sleep(5)

    print("no se pudo cargar la pagina")
    return None


def limpiar_texto(texto):
    if not texto:
        return None
    return re.sub(r"\s+", " ", texto).strip()

# EXTRAER LINKS OPINION

def extraer_links_opinion():
    links = set()

    PAGINAS = 300

    for pagina in range(1, PAGINAS + 1):

        if pagina == 1:
            url_pagina = SECCION
        else:
            url_pagina = f"{SECCION}{pagina}/"
            # equivale a: https://www.larazon.es/opinion/2/

        soup = get_soup(url_pagina, wait=4)

        if soup is None:
            print(f"pagina {pagina} fallida, sigo")
            continue

        encontrados_pagina = set()

        for a in soup.find_all("a", href=True):
            href = a["href"]

            url = urljoin(BASE, href)
            url = url.split("?")[0].split("#")[0]

            if (
                url.startswith(BASE)
                and "/opinion/" in url
                and url.endswith(".html")
            ):
                encontrados_pagina.add(url)

        antes = len(links)
        links.update(encontrados_pagina)
        nuevos = len(links) - antes

        print(
            f"pagina {pagina} | encontrados: {len(encontrados_pagina)} | nuevos: {nuevos} | total: {len(links)}"
        )

        if len(links) >= OBJETIVO:
            break

        time.sleep(2)

    return sorted(links)[:OBJETIVO]


# EXTRAER TITULO

def extraer_titulo(soup):
    meta = soup.find("meta", attrs={"property": "og:title"})
    if meta:
        return limpiar_texto(meta.get("content"))

    h1 = soup.find("h1")
    if h1:
        return limpiar_texto(h1.get_text())

    return None


# EXTRAER TEXTO

def extraer_texto(soup):
    article = soup.find("article")

    textos = []

    if article:
        for p in article.find_all("p"):
            t = limpiar_texto(p.get_text())
            if t and len(t) > 40:
                textos.append(t)

    if not textos:
        for p in soup.find_all("p"):
            t = limpiar_texto(p.get_text())
            if t and len(t) > 40:
                textos.append(t)

    if textos:
        return " ".join(dict.fromkeys(textos))

    return None

# EXTRAER ARTICULO

def extraer_articulo(url):
    soup = get_soup(url)

    if soup is None:
        return {
            "periodico": "la razon",
            "titulo": None,
            "texto": None,
            "url": url
        }

    titulo = extraer_titulo(soup)
    texto = extraer_texto(soup)

    return {
        "periodico": "la razon",
        "titulo": titulo,
        "texto": texto,
        "url": url
    }

# MAIN

print("\n--- RECOGIENDO LINKS ---")
links = extraer_links_opinion()

print("\nTOTAL LINKS:", len(links))

import os

BACKUP = "backup_larazon_opinion.csv"

if os.path.exists(BACKUP):
    df_backup = pd.read_csv(BACKUP)
    resultados = df_backup.to_dict("records")
    urls_ya_hechas = set(df_backup["url"])
    print(f"backup cargado: {len(resultados)} articulos")
else:
    resultados = []
    urls_ya_hechas = set()

print("\n--- PROCESANDO ARTICULOS ---")

for i, link in enumerate(links, 1):

    if link in urls_ya_hechas:
        print(f"\n[{i}/{len(links)}] ya procesado, salto")
        continue

    try:
        print(f"\n[{i}/{len(links)}] {link}")

        data = extraer_articulo(link)

        if data["titulo"] and data["texto"] and len(data["texto"]) > 200:
            resultados.append(data)
            urls_ya_hechas.add(link)
            print("añadido")
        else:
            print("descartado")

        print("total validos:", len(resultados))

        if len(resultados) % 25 == 0 and len(resultados) > 0:
            pd.DataFrame(resultados).drop_duplicates(subset=["url"]).to_csv(
                BACKUP,
                index=False,
                encoding="utf-8-sig"
            )
            print("backup guardado")

        if len(resultados) >= OBJETIVO:
            break

        time.sleep(3)

    except Exception as e:
        print("error:", e)

        try:
            driver.quit()
        except:
            pass

        driver = crear_driver()
        cookies_aceptadas = False
        time.sleep(5)


driver.quit()

# GUARDAR CSV

df = pd.DataFrame(resultados)

df = df.drop_duplicates(subset=["url"])
df = df[["periodico", "titulo", "texto", "url"]]

df.to_csv("data_larazon_opinion.csv", index=False, encoding="utf-8-sig")

print("\nTOTAL FINAL:", len(df))


--- RECOGIENDO LINKS ---

abriendo: https://www.larazon.es/opinion/ | intento 1
aceptando cookies...
pagina 1 | encontrados: 37 | nuevos: 37 | total: 37

abriendo: https://www.larazon.es/opinion/2/ | intento 1
pagina 2 | encontrados: 36 | nuevos: 36 | total: 73

abriendo: https://www.larazon.es/opinion/3/ | intento 1
pagina 3 | encontrados: 36 | nuevos: 36 | total: 109

abriendo: https://www.larazon.es/opinion/4/ | intento 1
pagina 4 | encontrados: 37 | nuevos: 37 | total: 146

abriendo: https://www.larazon.es/opinion/5/ | intento 1
pagina 5 | encontrados: 40 | nuevos: 40 | total: 186

abriendo: https://www.larazon.es/opinion/6/ | intento 1
pagina 6 | encontrados: 38 | nuevos: 38 | total: 224

abriendo: https://www.larazon.es/opinion/7/ | intento 1
pagina 7 | encontrados: 37 | nuevos: 37 | total: 261

abriendo: https://www.larazon.es/opinion/8/ | intento 1
pagina 8 | encontrados: 37 | nuevos: 37 | total: 298

abriendo: https://www.larazon.es/opinion/9/ | intento 1
pagina 9 | encontrad

In [11]:
df.shape
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2106 entries, 0 to 2105
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   periodico  2106 non-null   str  
 1   titulo     2106 non-null   str  
 2   texto      2106 non-null   str  
 3   url        2106 non-null   str  
dtypes: str(4)
memory usage: 65.9 KB


In [12]:
df.duplicated(subset=["url"]).sum()

np.int64(0)

In [13]:
df.duplicated(subset=["titulo"]).sum()

np.int64(4)

In [18]:
pd.set_option('display.max_colwidth', None)

df[df.duplicated(subset=["titulo"], keep=False)][["titulo", "url"]]

,titulo,url
206,Autónomos,https://www.larazon.es/opinion/autonomos_2025102068f5667651264f432a856c06.html
207,Autónomos,https://www.larazon.es/opinion/autonomos_202511286928df089261f37ec733695d.html
559,Contextos inventados,https://www.larazon.es/opinion/contextos-inventados_20251118691bb00f360d0840bce5eff1.html
560,Contextos inventados,https://www.larazon.es/opinion/contextos-inventados_20251118691bb011360d0840bce5eff5.html
1384,El mundo al revés,https://www.larazon.es/opinion/mundo-reves_2025102368f95afc5ddd5447a1bf96a8.html
1385,El mundo al revés,https://www.larazon.es/opinion/mundo-reves_20260108695eec85ea66eb7353296212.html
1583,Los precios no darán tregua,https://www.larazon.es/opinion/precios-daran-tregua_2026031769b8928f3156ba241d57b863.html
1584,Los precios no darán tregua,https://www.larazon.es/opinion/precios-daran-tregua_2026040469d03f84bfc2456bae1cd879.html


Al revisarse los títulos duplicados y comprobarse que algunos artículos aparecían bajo URLs distintas, se eliminaron los duplicados reales considerando conjuntamente el título y el texto del artículo, conservando únicamente una copia de cada contenido.